In [3]:
import os
from dataclasses import dataclass
from pathlib import Path


@dataclass(frozen=True)
class DataIngestionConfig:
    root_dir: Path
    source_URL: str
    local_data_file: Path
    unzip_dir: Path

In [4]:
from NexText.constants import CONFIG_FILE_PATH, PARAMS_FILE_PATH
from NexText.utils.common import read_yaml, create_directories
from NexText.entity.config_entity import DataIngestionConfig


class ConfigurationManager:

    def __init__(
        self,
        config_file_path=CONFIG_FILE_PATH,
        params_file_path=PARAMS_FILE_PATH
    ):
        self.config = read_yaml(config_file_path)
        self.params = read_yaml(params_file_path)

        create_directories([self.config.artifacts_root])

    def get_data_ingestion_config(self) -> DataIngestionConfig:

        config = self.config.data_ingestion

        create_directories([config.root_dir])

        return DataIngestionConfig(
            root_dir=config.root_dir,
            source_URL=config.source_URL,
            local_data_file=config.local_data_file,
            unzip_dir=config.unzip_dir
        )

In [5]:
from NexText.constants import CONFIG_FILE_PATH, PARAMS_FILE_PATH

print("Config:", CONFIG_FILE_PATH)
print("Config exists:", CONFIG_FILE_PATH.exists())

print("Params:", PARAMS_FILE_PATH)
print("Params exists:", PARAMS_FILE_PATH.exists())

Config: D:\NLP Project\config\config.yaml
Config exists: True
Params: D:\NLP Project\params.yaml
Params exists: True


In [6]:
import os
import urllib.request as request
import zipfile

from NexText.logging import logger
from NexText.utils.common import get_size
from NexText.entity.config_entity import DataIngestionConfig


class DataIngestion:

    def __init__(self, config: DataIngestionConfig):
        self.config = config

    def download_file(self):
        """Download the dataset ZIP file if it doesn't already exist."""

        if not os.path.exists(self.config.local_data_file):

            filename, headers = request.urlretrieve(
                url=self.config.source_URL,
                filename=self.config.local_data_file
            )

            logger.info(
                f"{filename} downloaded successfully!\n"
                f"Download information:\n{headers}"
            )

        else:
            logger.info(
                f"File already exists. "
                f"Size: {get_size(str(self.config.local_data_file))}"
            )

    def extract_zip_file(self):
        """Extract the downloaded ZIP file."""

        unzip_path = self.config.unzip_dir

        os.makedirs(unzip_path, exist_ok=True)

        with zipfile.ZipFile(
            self.config.local_data_file,
            "r"
        ) as zip_ref:

            zip_ref.extractall(unzip_path)

        logger.info(
            f"ZIP file extracted successfully to: {unzip_path}"
        )

In [8]:
try:
    config = ConfigurationManager()

    data_ingestion_config = config.get_data_ingestion_config()

    data_ingestion = DataIngestion(
        config=data_ingestion_config
    )

    data_ingestion.download_file()
    data_ingestion.extract_zip_file()

except Exception as e:
    raise e